# Iron Deficiency — ML Baseline

Target: `iron_deficiency` (binary, 6.05% prevalence, 15.5:1 imbalance)  
Model module: `models/iron_deficiency_model.py`  
Priority: **recall** (minimise missed cases) — threshold=0.3

**Models trained:**
- Logistic Regression (`class_weight='balanced'`)
- XGBoost (`scale_pos_weight=15`)

**Sections**
1. Setup & Data Loading
2. Feature Preparation
3. Train/Test Split
4. Feature Correlations (sanity check)
5. Train Logistic Regression
6. Train XGBoost
7. ROC Curves
8. Metrics Comparison
9. Feature Importance (XGBoost)
10. Comparison with Anemia Combined Model
11. Save Models

---
## 1. Setup & Data Loading

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from sklearn.metrics import roc_curve, auc

from models.iron_deficiency_model import (
    load_data, prepare_features, split_data,
    build_lr_pipeline, build_xgb_pipeline,
    evaluate_model, save_model,
    ENCODED_FEATURE_NAMES, FEATURE_GROUPS,
    TARGET_COL,
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

DATA_PATH = os.path.join(os.path.abspath('..'), 'data', 'processed', 'nhanes_merged_adults_final.csv')
df = load_data(DATA_PATH)

---
## 2. Feature Preparation

In [ ]:
X, y = prepare_features(df)

---
## 3. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y)

---
## 4. Feature Correlations (sanity check)

Point-biserial r vs `iron_deficiency` target on the full dataset.  
Iron-specific labs (Group A) expected to dominate — confirming leakage risk.

In [ ]:
corr_rows = []
for col in ENCODED_FEATURE_NAMES:
    combined = pd.concat([X[col], y], axis=1).dropna()
    if len(combined) < 30:
        continue
    r, p = stats.pointbiserialr(combined[col], combined[TARGET_COL])
    corr_rows.append({'feature': col, 'r': round(r, 4), 'p': round(p, 4)})

corr_df = pd.DataFrame(corr_rows).sort_values('r', key=abs, ascending=False)
print(corr_df.to_string(index=False))

In [ ]:
plot_df = corr_df.sort_values('r', ascending=True)

group_colors = {
    'iron_specific_labs':  '#DD8452',
    'checkup_labs':        '#4C72B0',
    'female_reproductive': '#55A868',
}
feat_to_group = {f: g for g, feats in FEATURE_GROUPS.items() for f in feats}
bar_colors = [group_colors.get(feat_to_group.get(f, ''), '#aaaaaa') for f in plot_df['feature']]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(plot_df['feature'], plot_df['r'], color=bar_colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)

patches = [mpatches.Patch(color=c, label=g.replace('_', ' '))
           for g, c in group_colors.items()]
ax.legend(handles=patches, fontsize=9)
ax.set_xlabel('Point-biserial r')
ax.set_title('Feature Correlations with Iron Deficiency\n(by feature group)', fontsize=12)
plt.tight_layout()
plt.show()

---
## 5. Train Logistic Regression

In [ ]:
lr = build_lr_pipeline()
lr.fit(X_train, y_train)
lr_metrics = evaluate_model(lr, X_test, y_test, threshold=0.3, model_name='LR Iron Deficiency')

---
## 6. Train XGBoost

In [ ]:
xgb = build_xgb_pipeline()
xgb.fit(X_train, y_train)
xgb_metrics = evaluate_model(xgb, X_test, y_test, threshold=0.3, model_name='XGBoost Iron Deficiency')

---
## 7. ROC Curves

In [ ]:
lr_proba  = lr.predict_proba(X_test)[:, 1]
xgb_proba = xgb.predict_proba(X_test)[:, 1]

lr_fpr,  lr_tpr,  _ = roc_curve(y_test, lr_proba)
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_proba)

lr_auc  = auc(lr_fpr,  lr_tpr)
xgb_auc = auc(xgb_fpr, xgb_tpr)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(lr_fpr,  lr_tpr,  color='#4C72B0', lw=2, label=f'Logistic Regression  AUC={lr_auc:.4f}')
ax.plot(xgb_fpr, xgb_tpr, color='#DD8452', lw=2, label=f'XGBoost              AUC={xgb_auc:.4f}')
ax.plot([0,1],[0,1], 'k--', lw=1, label='Random')
ax.axvline(0.3, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Iron Deficiency', fontsize=13)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

---
## 8. Metrics Comparison

In [ ]:
metrics_df = pd.DataFrame([lr_metrics, xgb_metrics])
print(metrics_df[['model', 'threshold', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']].to_string(index=False))

In [ ]:
metric_keys = ['roc_auc', 'recall', 'precision', 'f1']
x = np.arange(len(metric_keys))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, [lr_metrics[m]  for m in metric_keys], width, label='LR',     color='#4C72B0', edgecolor='white')
ax.bar(x + width/2, [xgb_metrics[m] for m in metric_keys], width, label='XGBoost', color='#DD8452', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels([m.upper() for m in metric_keys])
ax.set_ylim(0, 1.0)
ax.set_ylabel('Score')
ax.set_title(f'LR vs XGBoost — Iron Deficiency (threshold=0.3)', fontsize=12)
ax.legend()

for bars in ax.containers:
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)

plt.tight_layout()
plt.show()

---
## 9. Feature Importance (XGBoost)

In [ ]:
importances = xgb.named_steps['clf'].feature_importances_
imp_df = pd.DataFrame({'feature': ENCODED_FEATURE_NAMES, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=True)

feat_to_group = {f: g for g, feats in FEATURE_GROUPS.items() for f in feats}
group_colors = {
    'iron_specific_labs':  '#DD8452',
    'checkup_labs':        '#4C72B0',
    'female_reproductive': '#55A868',
}
bar_colors = [group_colors.get(feat_to_group.get(f, ''), '#aaaaaa') for f in imp_df['feature']]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(imp_df['feature'], imp_df['importance'], color=bar_colors, edgecolor='white')
patches = [mpatches.Patch(color=c, label=g.replace('_', ' '))
           for g, c in group_colors.items()]
ax.legend(handles=patches, fontsize=9)
ax.set_xlabel('Feature Importance (gain)')
ax.set_title('XGBoost Feature Importance — Iron Deficiency', fontsize=12)
plt.tight_layout()
plt.show()

---
## 10. Comparison with Anemia Combined Model

> Results below compare the iron deficiency models against the previously trained anemia combined model.  
> Both target conditions share the same NHANES dataset and use the same train/test seed (random_state=42).

| Model | Target | Prevalence | AUC-ROC | Recall (t=0.3) | F1 (t=0.3) |
|-------|--------|------------|---------|----------------|------------|
| LR Combined | Anemia | 4.81% | ~0.82 | ~0.76 | ~0.22 |
| XGB Combined | Anemia | 4.81% | ~0.84 | ~0.72 | ~0.26 |
| **LR Iron Def** | **Iron Deficiency** | **6.05%** | *see above* | *see above* | *see above* |
| **XGB Iron Def** | **Iron Deficiency** | **6.05%** | *see above* | *see above* | *see above* |

**Key differences:**
- Iron deficiency has stronger iron-specific lab features (r up to 0.52 vs anemia's hematology features)
- However, Group A (iron-specific) features introduce **leakage risk** — high AUC with those features may reflect label reproduction rather than true generalisation
- For checkup-constrained deployment (Group B + C only), expected AUC will be considerably lower — closer to the anemia model range
- Iron deficiency is more gender-skewed: `gender_female` r=+0.20 vs anemia where gender plays a smaller role in the checkup-only model
- All questionnaire features were dropped for iron deficiency (subclinical without anaemia), whereas they contributed modestly to the anemia combined model

In [ ]:
# Visual comparison — fill in anemia combined values if available
# Reference values from ml-anemia-combined.ipynb runs
anemia_lr_ref  = {'model': 'LR Anemia Combined',  'roc_auc': None, 'recall': None, 'precision': None, 'f1': None}
anemia_xgb_ref = {'model': 'XGB Anemia Combined', 'roc_auc': None, 'recall': None, 'precision': None, 'f1': None}

# Iron deficiency — from this run
iron_lr_ref  = {k: lr_metrics[k]  for k in ('roc_auc', 'recall', 'precision', 'f1')}
iron_xgb_ref = {k: xgb_metrics[k] for k in ('roc_auc', 'recall', 'precision', 'f1')}

iron_lr_ref['model']  = 'LR Iron Deficiency'
iron_xgb_ref['model'] = 'XGB Iron Deficiency'

compare_df = pd.DataFrame([iron_lr_ref, iron_xgb_ref])
print('Iron Deficiency Model Results:')
print(compare_df.to_string(index=False))
print('\nNote: Compare against ml-anemia-combined.ipynb results for side-by-side view.')

---
## 11. Save Models

In [ ]:
lr_path  = save_model(lr,  'iron_deficiency_lr',  lr_metrics)
xgb_path = save_model(xgb, 'iron_deficiency_xgb', xgb_metrics)

print('\nFinal summary:')
print(pd.DataFrame([lr_metrics, xgb_metrics])[['model','threshold','accuracy','precision','recall','f1','roc_auc']].to_string(index=False))

---
## Model Notes

### Deployment warning — Group A leakage
The iron-specific lab features (ferritin, serum iron, TIBC, transferrin saturation, UIBC, transferrin receptor) are almost certainly part of the clinical criteria used to assign the `iron_deficiency` label in NHANES. Including them achieves high AUC but the model is partially re-deriving the label definition, not predicting from upstream symptoms.

**For a realistic screening tool** (where iron deficiency is *unknown*): retrain using Group B + C only.

### Class imbalance strategy
- LR: `class_weight='balanced'` — sample weights inversely proportional to class frequency
- XGBoost: `scale_pos_weight=15` — reweights positive class gradient
- Threshold: 0.3 (vs default 0.5) — shifts operating point toward higher recall at cost of precision

### Next steps
1. Retrain on Group B + C only for leakage-free evaluation
2. Tune threshold on validation set (precision-recall curve)
3. Consider SMOTE or other oversampling for training set